In [ ]:
!pip install pyspark py4j

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import to_date, mean, col, coalesce

spark = SparkSession.builder.appName("Read CSV Example").getOrCreate()

# Чтение CSV-файла
df = spark.read.csv("/content/drive/MyDrive/Colab Notebooks/weather_data.csv", header=True, inferSchema=True)

df = df.select(df.station_id, to_date(df.date, "yyyy-MM-dd").alias("date"), df.temperature, df.precipitation, df.wind_speed)

#Заполните пропущенные значения, если такие в csv файле есть (например, используя средние значения по метеостанциям)
station_means = df.groupBy("station_id").agg(
  mean("temperature").alias("mean_temp"),
  mean("precipitation").alias("mean_precip"),
  mean("wind_speed").alias("mean_wind")
)

df = df.join(station_means, on="station_id", how="left")

df = df.select(
    col("station_id"),
    col("date"),
    coalesce(col("temperature"), col("mean_temp")).alias("temperature"),
    coalesce(col("precipitation"), col("mean_precip")).alias("precipitation"),
    coalesce(col("wind_speed"), col("mean_wind")).alias("wind_speed")
)

df = df.drop("mean_temp", "mean_precip", "mean_wind")

df.createOrReplaceTempView("weather_data")

top_hot_df = spark.sql("""
SELECT date, temperature
FROM weather_data
order by 2 desc
limit 5
""")

most_precipitation_df = spark.sql("""
SELECT station_id, sum(precipitation)
FROM weather_data
where year(date) = (select max(year(date)) from weather_data)
group by station_id, year(date)
order by 2 desc
limit 1
""")

avg_temp_df = spark.sql("""
SELECT month(date) as month, avg(temperature)
FROM weather_data
group by month(date)
order by 1
""")

# Показ результатов
top_hot_df.show() # Найдите топ-5 самых жарких дней за все время наблюдений.
most_precipitation_df.show() # Найдите метеостанцию с наибольшим количеством осадков за последний год.
avg_temp_df.show()# Подсчитайте среднюю температуру по месяцам за все время наблюдений.


+----------+------------------+
|      date|       temperature|
+----------+------------------+
|2021-08-20|39.982828249354846|
|2023-12-02| 39.96797489293784|
|2022-03-28|  39.8246894248997|
|2019-02-11| 39.76737697836647|
|2020-06-10| 39.69147838355929|
+----------+------------------+

+----------+------------------+
|station_id|sum(precipitation)|
+----------+------------------+
| station_5| 642.9302626767898|
+----------+------------------+

+-----+------------------+
|month|  avg(temperature)|
+-----+------------------+
|    1|11.356518462550754|
|    2| 9.067229891101926|
|    3| 7.244080205633994|
|    4|12.024529009744693|
|    5| 9.902883346912718|
|    6|13.421092297254138|
|    7|6.1857183016954576|
|    8|  10.9678002814186|
|    9| 9.596744236573942|
|   10|  9.09884344821895|
|   11| 7.265889994697494|
|   12|11.218592100674337|
+-----+------------------+

